# Chapter 4 — Full GPT Architecture

**Goal:** Put all components together into a complete GPT model.

### Architecture overview
```
Token IDs
   ↓
EmbeddingLayer (token + positional)
   ↓
TransformerBlock × N
   ↓
LayerNorm
   ↓
LM Head (Linear → vocab_size)
   ↓
Logits
```

In [ ]:
import sys, torch
sys.path.insert(0, '..')
from src.model.gpt import GPT, GPTConfig

## 1. Build a Tiny GPT

In [ ]:
cfg = GPTConfig(
    vocab_size=50257,
    context_length=64,
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    dropout=0.1,
)

model = GPT(cfg)
print(f'Parameters: {model.num_parameters():,}')
print(model)

## 2. Forward Pass

In [ ]:
batch_size, seq_len = 2, 16
x = torch.randint(0, cfg.vocab_size, (batch_size, seq_len))

logits = model(x)
print(f'Input shape:  {x.shape}')          # (2, 16)
print(f'Logits shape: {logits.shape}')      # (2, 16, 50257)

## 3. Autoregressive Generation

In [ ]:
# Start with 5 random token IDs as a prompt
prompt = torch.randint(0, cfg.vocab_size, (1, 5))
generated = model.generate(prompt, max_new_tokens=20, top_k=10)

print(f'Prompt length:    {prompt.shape[1]}')
print(f'Generated length: {generated.shape[1]}')
print(f'New tokens: {generated[0, 5:].tolist()}')

## 4. Compare GPT-2 sizes

In [ ]:
sizes = {
    'Small  (117M)': GPTConfig.small(),
    'Medium (345M)': GPTConfig.medium(),
    'Large  (762M)': GPTConfig.large(),
}
for name, c in sizes.items():
    m = GPT(c)
    print(f'{name}: {m.num_parameters():>12,} params   d_model={c.embed_dim}  layers={c.num_layers}')

## 5. Exercises
1. Why do we tie the token embedding weights with the LM head?
2. What is the memory footprint (in MB) of GPT-2 Small in float32?
3. Experiment with `temperature` in `.generate()` — how does it affect randomness?